# Projeto G2 - Turismo no Brasil

Este notebook apresenta uma análise exploratória da base simulada de turismo no Brasil entre 2015 e 2024. O objetivo é investigar padrões de fluxo turístico, sazonalidade, comparação regional, ocupação hoteleira, clima e impacto econômico.

## 1. Contextualização

O turismo é uma atividade econômica relevante porque movimenta o setor de serviços, gera empregos, estimula investimentos em infraestrutura e contribui para o desenvolvimento regional. A análise de dados permite identificar destinos mais procurados, períodos de alta temporada e diferenças econômicas entre regiões.

## 2. Explicação da Base

A base contém informações por ano, mês, região, estado e cidade turística. As principais variáveis são quantidade de turistas, turistas estrangeiros, ocupação hoteleira, gasto médio, faturamento turístico, eventos realizados, temperatura média e nível de temporada.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option('display.float_format', lambda valor: f'{valor:,.2f}')

MESES = {
    1: 'Jan', 2: 'Fev', 3: 'Mar', 4: 'Abr', 5: 'Mai', 6: 'Jun',
    7: 'Jul', 8: 'Ago', 9: 'Set', 10: 'Out', 11: 'Nov', 12: 'Dez'
}
CORES = ['#2563EB', '#059669', '#F59E0B', '#DC2626', '#7C3AED']

## 3. Leitura dos Dados

In [ ]:
caminho = '../dados/simulacao_turismo_brasil.csv'
df = pd.read_csv(caminho)

print(f'Linhas: {df.shape[0]}')
print(f'Colunas: {df.shape[1]}')
df.head()

## 4. Limpeza e Preparação

Nesta etapa são verificados tipos, valores ausentes e consistência temporal.

In [ ]:
df.info()
df.isna().sum()

In [ ]:
df['data'] = pd.to_datetime(df['data'], errors='coerce')

colunas_numericas = [
    'ano', 'mes', 'turistas', 'turistas_estrangeiros', 'ocupacao_hoteleira',
    'gasto_medio', 'faturamento_turismo', 'eventos_realizados', 'temperatura_media'
]

for coluna in colunas_numericas:
    df[coluna] = pd.to_numeric(df[coluna], errors='coerce')

df = df.dropna(subset=['ano', 'mes', 'data', 'turistas', 'faturamento_turismo']).copy()
df['ano'] = df['ano'].astype(int)
df['mes'] = df['mes'].astype(int)

df[['ano', 'mes', 'data']].head()

## 5. Engenharia de Atributos

Foram criadas variáveis auxiliares para facilitar as análises temporais e econômicas.

In [ ]:
df['mes_nome'] = df['mes'].map(MESES)
df['ano_mes'] = pd.to_datetime(df['ano'].astype(str) + '-' + df['mes'].astype(str).str.zfill(2) + '-01')
df['participacao_estrangeiros'] = np.where(df['turistas'] > 0, df['turistas_estrangeiros'] / df['turistas'] * 100, 0)
df['receita_por_turista'] = np.where(df['turistas'] > 0, df['faturamento_turismo'] / df['turistas'], 0)

df[['cidade', 'ano_mes', 'participacao_estrangeiros', 'receita_por_turista']].head()

## 6. KPIs

In [ ]:
total_turistas = df['turistas'].sum()
receita_total = df['faturamento_turismo'].sum()
ocupacao_media = df['ocupacao_hoteleira'].mean()
gasto_medio_ponderado = np.average(df['gasto_medio'], weights=df['turistas'])

cidade_mais_visitada = df.groupby('cidade')['turistas'].sum().sort_values(ascending=False).head(1)
regiao_mais_movimentada = df.groupby('regiao')['faturamento_turismo'].sum().sort_values(ascending=False).head(1)

kpis = pd.DataFrame({
    'Indicador': [
        'Total de turistas', 'Receita total do turismo', 'Ocupação hoteleira média',
        'Gasto médio ponderado', 'Cidade mais visitada', 'Região mais movimentada'
    ],
    'Valor': [
        total_turistas, receita_total, ocupacao_media, gasto_medio_ponderado,
        cidade_mais_visitada.index[0], regiao_mais_movimentada.index[0]
    ]
})

kpis

## 7. Visualizações

### 7.1 Evolução temporal do turismo

In [ ]:
temporal = df.groupby('ano_mes', as_index=False).agg(
    turistas=('turistas', 'sum'),
    faturamento_turismo=('faturamento_turismo', 'sum')
)

fig = make_subplots(specs=[[{'secondary_y': True}]])
fig.add_trace(go.Scatter(x=temporal['ano_mes'], y=temporal['turistas'], mode='lines+markers', name='Turistas'), secondary_y=False)
fig.add_trace(go.Scatter(x=temporal['ano_mes'], y=temporal['faturamento_turismo'], mode='lines', name='Faturamento'), secondary_y=True)
fig.update_layout(title='Evolução mensal de turistas e faturamento', template='plotly_white', hovermode='x unified')
fig.update_yaxes(title_text='Turistas', secondary_y=False)
fig.update_yaxes(title_text='Faturamento (R$)', secondary_y=True)
fig.show()

### 7.2 Comparação entre regiões

In [ ]:
regioes = df.groupby('regiao', as_index=False).agg(
    turistas=('turistas', 'sum'),
    faturamento_turismo=('faturamento_turismo', 'sum'),
    ocupacao_hoteleira=('ocupacao_hoteleira', 'mean')
).sort_values('faturamento_turismo', ascending=False)

px.bar(
    regioes,
    x='regiao',
    y='faturamento_turismo',
    color='turistas',
    title='Faturamento turístico por região',
    labels={'regiao': 'Região', 'faturamento_turismo': 'Faturamento (R$)', 'turistas': 'Turistas'},
    template='plotly_white'
).show()

regioes

### 7.3 Ranking de destinos turísticos

In [ ]:
top_cidades = df.groupby(['cidade', 'uf', 'regiao'], as_index=False).agg(
    turistas=('turistas', 'sum'),
    faturamento_turismo=('faturamento_turismo', 'sum')
).sort_values('turistas', ascending=False).head(15)

px.bar(
    top_cidades.sort_values('turistas'),
    x='turistas',
    y='cidade',
    color='regiao',
    orientation='h',
    title='Top 15 cidades por quantidade de turistas',
    labels={'turistas': 'Turistas', 'cidade': 'Cidade', 'regiao': 'Região'},
    template='plotly_white'
).show()

top_cidades

### 7.4 Heatmap mensal de sazonalidade

In [ ]:
sazonalidade = df.pivot_table(
    index='ano', columns='mes', values='turistas', aggfunc='sum', fill_value=0
).reindex(columns=range(1, 13), fill_value=0)
sazonalidade.columns = [MESES[mes] for mes in sazonalidade.columns]

px.imshow(
    sazonalidade,
    aspect='auto',
    color_continuous_scale='YlGnBu',
    text_auto='.2s',
    title='Heatmap mensal de turistas',
    labels=dict(x='Mês', y='Ano', color='Turistas')
).show()

### 7.5 Dispersão turistas x faturamento

In [ ]:
destinos = df.groupby(['cidade', 'uf', 'regiao'], as_index=False).agg(
    turistas=('turistas', 'sum'),
    faturamento_turismo=('faturamento_turismo', 'sum'),
    ocupacao_hoteleira=('ocupacao_hoteleira', 'mean'),
    gasto_medio=('gasto_medio', 'mean')
)

px.scatter(
    destinos,
    x='turistas',
    y='faturamento_turismo',
    color='regiao',
    size='ocupacao_hoteleira',
    hover_name='cidade',
    title='Relação entre turistas e faturamento por destino',
    labels={'turistas': 'Turistas', 'faturamento_turismo': 'Faturamento (R$)', 'regiao': 'Região'},
    template='plotly_white'
).show()

### 7.6 Relação clima x turismo e ocupação hoteleira

In [ ]:
px.scatter(
    df,
    x='temperatura_media',
    y='turistas',
    color='nivel_temporada',
    size='eventos_realizados',
    hover_data=['ano', 'mes_nome', 'cidade', 'regiao'],
    title='Relação clima x turismo',
    labels={'temperatura_media': 'Temperatura média (°C)', 'turistas': 'Turistas', 'nivel_temporada': 'Temporada'},
    template='plotly_white'
).show()

px.box(
    df,
    x='nivel_temporada',
    y='ocupacao_hoteleira',
    color='nivel_temporada',
    title='Distribuição da ocupação hoteleira por temporada',
    labels={'nivel_temporada': 'Temporada', 'ocupacao_hoteleira': 'Ocupação hoteleira (%)'},
    template='plotly_white'
).show()

## 8. Destinos com Maior Crescimento

A comparação entre 2015 e 2024 ajuda a identificar cidades com expansão relevante no fluxo turístico.

In [ ]:
ano_inicial = df['ano'].min()
ano_final = df['ano'].max()

crescimento = (
    df[df['ano'].isin([ano_inicial, ano_final])]
    .groupby(['cidade', 'ano'])['turistas']
    .sum()
    .unstack(fill_value=0)
)

crescimento = crescimento[crescimento[ano_inicial] > 0].copy()
crescimento['crescimento_percentual'] = (crescimento[ano_final] - crescimento[ano_inicial]) / crescimento[ano_inicial] * 100
crescimento['crescimento_absoluto'] = crescimento[ano_final] - crescimento[ano_inicial]
crescimento = crescimento.sort_values('crescimento_percentual', ascending=False).head(10)

px.bar(
    crescimento.reset_index().sort_values('crescimento_percentual'),
    x='crescimento_percentual',
    y='cidade',
    orientation='h',
    title=f'Destinos com maior crescimento de turistas ({ano_inicial} x {ano_final})',
    labels={'crescimento_percentual': 'Crescimento (%)', 'cidade': 'Cidade'},
    template='plotly_white'
).show()

crescimento

## 9. Tabela Dinâmica

In [ ]:
tabela_dinamica = pd.pivot_table(
    df,
    index=['regiao', 'uf', 'cidade'],
    columns='nivel_temporada',
    values='turistas',
    aggfunc='sum',
    fill_value=0
)
tabela_dinamica['Total turistas'] = tabela_dinamica.sum(axis=1)
tabela_dinamica.sort_values('Total turistas', ascending=False).head(20)

## 10. Interpretação dos Resultados

A análise permite observar quais destinos concentram maior fluxo turístico, quais regiões movimentam mais faturamento e quais meses apresentam maior intensidade de viagens. A comparação entre turistas e faturamento também ajuda a avaliar se o impacto econômico acompanha o volume de visitantes.

A ocupação hoteleira e a classificação de temporada indicam pressões diferentes sobre o setor de hospedagem. Já a relação entre temperatura, eventos e turistas pode sugerir padrões ligados a clima e calendário turístico.

## 11. Conclusão

O turismo no Brasil, mesmo em uma base simulada, demonstra grande potencial analítico. Os indicadores de fluxo, faturamento, gasto médio e hotelaria permitem avaliar a relevância econômica do setor e apoiar decisões sobre infraestrutura, promoção de destinos e planejamento regional. O dashboard em Streamlit complementa este notebook ao permitir exploração interativa dos filtros e visualizações.